# Proyecto 2: Clasificación de tejidos en imágenes de cáncer colorrectal

### Visión artificial

Universidad de Medellín - Facultad de Ingenierías - Ingeniería de Sistemas

**Curso:** Procesamiento de Datos

**Profesor:** Antonio Jesús Tamayo Herrera, PhD

**Integrantes:** Sebastián Quintero Granada, (compañero 2), (compañero 3)

En este proyecto entrenamos un modelo que, a partir de una imagen de tejido de cáncer colorrectal vista al
microscopio, identifica qué tipo de tejido es. La base se carga desde TensorFlow Datasets, hacemos un análisis
exploratorio de las imágenes y aplicamos fine tuning sobre dos redes preentrenadas en ImageNet, VGG16 y ResNet50,
probando varias combinaciones de hiperparámetros. Al final evaluamos el mejor modelo en un conjunto de prueba que
no se usó ni para entrenar ni para escoger el modelo.

Se recomienda correrlo en Google Colab con GPU (Entorno de ejecución → Cambiar tipo de entorno → GPU T4).

## 1. Librerías

In [ ]:
# TFDS necesita este paquete en Python 3.13 (versión actual de Colab) y no lo instala solo
!pip install -q importlib_resources

import itertools
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

keras.utils.set_random_seed(42)
tf.get_logger().setLevel('ERROR')  # oculta avisos internos de TensorFlow
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Descripción de la base de datos

Usamos la base [`colorectal_histology`](https://www.tensorflow.org/datasets/catalog/colorectal_histology) del
catálogo de TensorFlow Datasets, publicada por Kather et al. (2016). Tiene 5.000 imágenes de tejido de cáncer
colorrectal vistas al microscopio. Cada imagen es un recorte de 150 × 150 píxeles en color (RGB), tomado de
muestras teñidas con hematoxilina y eosina (H&E), que es la tinción que se usa normalmente en patología: los
núcleos de las células quedan morados y el resto del tejido queda rosado.

**Dominio y utilidad.** Es un problema de patología digital. Una muestra de tumor no tiene solo células
cancerosas, también tiene tejido conectivo, células inmunes, grasa, moco y zonas vacías. Un modelo que reconozca
el tipo de tejido en cada recorte permite medir qué parte de la muestra es tumor, estroma o células inmunes, algo
que normalmente el patólogo estima a ojo y que sirve de apoyo para el diagnóstico.

**Variable de salida.** La etiqueta `label` indica el tipo de tejido y toma 8 valores:

- 0: Epitelio tumoral
- 1: Estroma simple
- 2: Estroma complejo (estroma con células tumorales o inmunes sueltas)
- 3: Células inmunes
- 4: Detritos y moco
- 5: Glándulas mucosas (mucosa sana)
- 6: Tejido adiposo
- 7: Fondo (zona sin tejido)

**Tipo de problema.** Clasificación multiclase (8 clases) con aprendizaje supervisado. Lo resolvemos con redes
neuronales convolucionales usando fine tuning, y como hay 8 clases medimos el desempeño con accuracy y con F1
macro (promedio del F1 de todas las clases).

## 3. Carga de los datos

In [ ]:
datos, info = tfds.load('colorectal_histology', split='train', as_supervised=True,
                        batch_size=-1, with_info=True)
X, y = tfds.as_numpy(datos)

clases = ['Epitelio tumoral', 'Estroma simple', 'Estroma complejo', 'Células inmunes',
          'Detritos y moco', 'Glándulas mucosas', 'Tejido adiposo', 'Fondo']

print('Imágenes:', X.shape, X.dtype)
print('Etiquetas:', y.shape, '| número de clases:', info.features['label'].num_classes)
print('Nombres originales:', info.features['label'].names)

La base solo trae la partición `train`, así que más adelante la dividimos en entrenamiento, validación y prueba.
Como son solo 5.000 imágenes pequeñas, las cargamos completas en memoria como arreglos de NumPy.

## 4. Análisis exploratorio

### 4.1 Cantidad de imágenes por clase

In [ ]:
conteo = np.bincount(y)
plt.figure(figsize=(10, 4))
plt.bar(clases, conteo, color='#2a78d6')
for i, n in enumerate(conteo):
    plt.text(i, n + 5, n, ha='center')
plt.title('Imágenes por clase')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

Las 8 clases tienen la misma cantidad de imágenes (625 cada una), así que la base está balanceada. No hace falta
aplicar ninguna técnica de balanceo y la accuracy sí es una métrica representativa, aunque igual reportamos el F1 macro.

### 4.2 Ejemplos de cada clase

In [ ]:
fig, axes = plt.subplots(8, 5, figsize=(10, 16))
for c in range(8):
    indices = np.random.choice(np.where(y == c)[0], 5, replace=False)
    for j, idx in enumerate(indices):
        axes[c, j].imshow(X[idx])
        axes[c, j].set_xticks([])
        axes[c, j].set_yticks([])
    axes[c, 0].set_ylabel(clases[c], rotation=0, ha='right', va='center')
plt.tight_layout()
plt.show()

Se nota la tinción H&E: el epitelio tumoral y las células inmunes se ven más morados y oscuros porque tienen muchos
núcleos juntos, mientras que el estroma es más rosado. El tejido adiposo y el fondo son casi blancos. Algunas clases
se parecen bastante, por ejemplo el estroma simple y el complejo, porque la diferencia está en unas pocas células
sueltas dentro del tejido.

### 4.3 Color promedio e histogramas de color

In [ ]:
# Imagen promedio de cada clase (NumPy)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
for c in range(8):
    axes[c].imshow(X[y == c].mean(axis=0).astype('uint8'))
    axes[c].set_title(clases[c], fontsize=9)
    axes[c].axis('off')
plt.suptitle('Imagen promedio por clase')
plt.tight_layout()
plt.show()

# Histograma de cada canal por clase (OpenCV), juntando todas las imágenes de la clase
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
for c, ax in enumerate(axes.ravel()):
    pixeles = np.ascontiguousarray(X[y == c].reshape(-1, 150, 3))
    for canal, color in enumerate(['red', 'green', 'blue']):
        hist = cv2.calcHist([pixeles], [canal], None, [256], [0, 256])
        ax.plot(hist / hist.sum(), color=color)
    ax.set_title(clases[c])
for ax in axes[1]:
    ax.set_xlabel('Intensidad')
plt.suptitle('Histogramas RGB por clase')
plt.tight_layout()
plt.show()

La imagen promedio muestra el color dominante de cada tejido (la forma se pierde porque los recortes no están
alineados). En los histogramas, el fondo y el tejido adiposo tienen casi todos sus píxeles en valores altos (claros),
mientras que las demás clases tienen distribuciones parecidas entre sí. Por eso el color solo no alcanza para separar
todas las clases, y tiene sentido usar una CNN, que además aprende la textura del tejido.

## 5. Preparación de los datos

Dividimos la base de forma estratificada en entrenamiento (70 %), validación (15 %) y prueba (15 %). Validación se usa
para comparar las combinaciones de la malla y prueba solo al final, con el mejor modelo.

El preprocesamiento de cada red (`preprocess_input`) y el aumento de datos van dentro del modelo. Como aumento usamos
volteos y rotaciones, que no cambian el tipo de tejido.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print('Entrenamiento:', len(X_train), '| Validación:', len(X_val), '| Prueba:', len(X_test))

## 6. Modelo y fine tuning

Entrenar VGG16 o ResNet50 desde cero con 3.500 imágenes llevaría a sobreajuste, así que aprovechamos sus pesos
entrenados en ImageNet. A cada red le quitamos el clasificador original y le ponemos uno nuevo:
GlobalAveragePooling, una capa densa de 256 neuronas con dropout y la salida softmax de 8 clases.

El entrenamiento tiene dos fases:

1. Con la red base congelada, solo se entrena el clasificador nuevo.
2. Se descongela el último bloque convolucional (`block5` en VGG16 y `conv5` en ResNet50) y se sigue entrenando
   con un learning rate 10 veces menor. Este es el fine tuning como tal.

En las dos fases usamos early stopping sobre la pérdida de validación.

In [ ]:
REDES = {
    'VGG16': (keras.applications.VGG16, keras.applications.vgg16.preprocess_input),
    'ResNet50': (keras.applications.ResNet50, keras.applications.resnet50.preprocess_input),
}
EPOCAS = 5  # máximo por fase


def construir_modelo(arquitectura):
    red, preprocesar = REDES[arquitectura]
    base = red(include_top=False, weights='imagenet', input_shape=(150, 150, 3))
    base.trainable = False

    entrada = keras.Input(shape=(150, 150, 3))
    x = layers.RandomFlip('horizontal_and_vertical')(entrada)
    x = layers.RandomRotation(0.25, fill_mode='reflect')(x)
    x = preprocesar(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    salida = layers.Dense(8, activation='softmax')(x)
    return keras.Model(entrada, salida), base


def entrenar(arquitectura, batch_size, lr):
    modelo, base = construir_modelo(arquitectura)
    parada = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]

    # Fase 1: base congelada
    modelo.compile(optimizer=keras.optimizers.Adam(lr), loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
    h1 = modelo.fit(X_train, y_train, batch_size=batch_size, epochs=EPOCAS,
                    validation_data=(X_val, y_val), callbacks=parada, verbose=0)

    # Fase 2: fine tuning del último bloque
    base.trainable = True
    for capa in base.layers:
        capa.trainable = capa.name.startswith(('block5', 'conv5'))
    modelo.compile(optimizer=keras.optimizers.Adam(lr / 10), loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
    h2 = modelo.fit(X_train, y_train, batch_size=batch_size, epochs=EPOCAS,
                    validation_data=(X_val, y_val), callbacks=parada, verbose=0)

    historia = {k: h1.history[k] + h2.history[k] for k in h1.history}
    return modelo, historia, len(h1.history['loss'])

## 7. Malla de hiperparámetros

Probamos todas las combinaciones de arquitectura (VGG16 y ResNet50), batch size (16 y 32) y learning rate inicial
(0,001 y 0,0001), en total 8 modelos. Nos quedamos con el que tenga mayor F1 macro en validación.
Esta celda es la más demorada.

In [ ]:
malla = {'arquitectura': ['VGG16', 'ResNet50'], 'batch_size': [16, 32], 'learning_rate': [1e-3, 1e-4]}

resultados, historias = [], {}
mejor_f1 = -1
for arq, bs, lr in itertools.product(*malla.values()):
    nombre = f'{arq} | batch {bs} | lr {lr}'
    inicio = time.time()
    modelo, historia, epocas_fase1 = entrenar(arq, bs, lr)

    pred_val = modelo.predict(X_val, verbose=0).argmax(axis=1)
    acc = accuracy_score(y_val, pred_val)
    f1 = f1_score(y_val, pred_val, average='macro')
    minutos = (time.time() - inicio) / 60

    resultados.append({'modelo': nombre, 'arquitectura': arq, 'batch_size': bs, 'learning_rate': lr,
                       'accuracy_val': acc, 'f1_macro_val': f1, 'minutos': minutos})
    historias[nombre] = (historia, epocas_fase1)
    print(f'{nombre:<30} accuracy={acc:.4f}  F1 macro={f1:.4f}  ({minutos:.1f} min)')

    if f1 > mejor_f1:
        mejor_f1, mejor_nombre, mejor_modelo = f1, nombre, modelo

In [ ]:
tabla = pd.DataFrame(resultados).sort_values('f1_macro_val', ascending=False).reset_index(drop=True)
tabla.round(4)

In [ ]:
orden = tabla.sort_values('f1_macro_val')
colores = ['#2a78d6' if a == 'VGG16' else '#eb6834' for a in orden['arquitectura']]

plt.figure(figsize=(9, 4.5))
plt.barh(orden['modelo'], orden['f1_macro_val'], color=colores)
for i, v in enumerate(orden['f1_macro_val']):
    plt.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)
plt.xlim(max(0, orden['f1_macro_val'].min() - 0.05), 1)
plt.xlabel('F1 macro en validación')
plt.title('Resultados de la malla (azul: VGG16, naranja: ResNet50)')
plt.tight_layout()
plt.show()

*(Completar después de correr: qué combinación ganó, si en general fue mejor VGG16 o ResNet50 y qué efecto
tuvieron el batch size y el learning rate.)*

## 8. Mejor modelo

In [ ]:
historia, epocas_fase1 = historias[mejor_nombre]
print(f'Mejor modelo: {mejor_nombre}  (F1 macro en validación: {mejor_f1:.4f})')

epocas = range(1, len(historia['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metrica, titulo in zip(axes, ['loss', 'accuracy'], ['Pérdida', 'Accuracy']):
    ax.plot(epocas, historia[metrica], marker='o', label='Entrenamiento')
    ax.plot(epocas, historia['val_' + metrica], marker='o', label='Validación')
    ax.axvline(epocas_fase1 + 0.5, color='gray', linestyle='--', label='Inicio fine tuning')
    ax.set_title(titulo)
    ax.set_xlabel('Época')
    ax.legend()
plt.tight_layout()
plt.show()

Ahora sí evaluamos en el conjunto de prueba, que no se había usado hasta este punto.

In [ ]:
pred_test = mejor_modelo.predict(X_test, verbose=0).argmax(axis=1)

print(f'Accuracy en prueba: {accuracy_score(y_test, pred_test):.4f}')
print(f'F1 macro en prueba: {f1_score(y_test, pred_test, average="macro"):.4f}')
print()
print(classification_report(y_test, pred_test, target_names=clases, digits=4))

In [ ]:
mc = confusion_matrix(y_test, pred_test)
plt.figure(figsize=(8, 6.5))
sns.heatmap(mc, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=clases, yticklabels=clases)
plt.xlabel('Predicción')
plt.ylabel('Clase real')
plt.title('Matriz de confusión en prueba')
plt.tight_layout()
plt.show()

In [ ]:
# Algunas predicciones del conjunto de prueba (verde: acierto, rojo: error)
indices = np.random.choice(len(X_test), 12, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for ax, i in zip(axes.ravel(), indices):
    ax.imshow(X_test[i])
    color = 'green' if pred_test[i] == y_test[i] else 'red'
    ax.set_title(f'Real: {clases[y_test[i]]}\nPred: {clases[pred_test[i]]}', fontsize=8, color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

*(Completar después de correr: resultados en prueba, qué clases reconoce mejor el modelo y entre cuáles se confunde.)*

## 9. Conclusiones

- El fine tuning permite aprovechar redes entrenadas con millones de imágenes de ImageNet en un problema de
  histología con solo 3.500 imágenes de entrenamiento. Aunque ImageNet no tiene imágenes de tejidos, los bordes,
  colores y texturas que aprendieron las primeras capas también sirven aquí.
- Entrenar en dos fases evita dañar los pesos preentrenados al principio, cuando el clasificador nuevo todavía es
  aleatorio. En las curvas del mejor modelo se puede ver qué pasa al empezar el fine tuning.
- La malla muestra que el batch size y el learning rate también cambian el resultado, así que comparar las dos redes
  con una sola configuración no habría sido justo.
- *(Completar: mejor modelo y sus métricas en prueba.)*

Como limitación, todas las imágenes vienen de un mismo estudio, y muestras de otros laboratorios pueden tener
colores distintos por la tinción o el escáner. El modelo es un apoyo y no reemplaza el criterio del patólogo.

**Referencia:** Kather, J. N. et al. (2016). Multi-class texture analysis in colorectal cancer histology.
*Scientific Reports*, 6, 27988.